<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="50%"></a>
</p>

<style>
.lx-table {
  margin: 1.5em auto;
  text-align: left;
}

.lx-table caption {
  caption-side: top;
  font-size: 0.9em;
  margin-bottom: 0.6em;
  text-align: center;
}

.lx-figure {
  margin: 1.5em auto;
  text-align: center;
}

.lx-figure figcaption {
  font-size: 0.9em;
  margin-top: 0.6em;
  text-align: center;
}

table {
  margin: 0 auto 1.5em;
}

p:has(> a[id^='table-']) {
  margin: 0;
}

p:has(> a[id^='table-']) + p,
a[id^='table-'] + p {
  font-size: 0.9em;
  margin: 1.5em 0 0.6em;
  text-align: center;
}

p.lx-figure {
  margin: 1.5em auto 0;
}

p.lx-figure + p {
  font-size: 0.9em;
  margin: 0.6em 0 1.5em;
  text-align: center;
}
</style>

# ROS 2 Discovery, Domains, and Names

ROS 2 nodes discover one another through middleware. This notebook explains how discovery works on the Duckiedrone, along with graph domains and ROS 2 name resolution.

## Discovery through middleware

Middleware is the software layer that lets separate ROS 2 nodes find one another and exchange messages. Compatible participants must be able to reach one another over the network and use the same domain identifier (ID). The `ROS_DOMAIN_ID` environment variable contains that identifier, a number that separates independent ROS 2 graphs sharing a network; participants with different values do not join the same graph.

The Duckiedrone uses [Zenoh](https://zenoh.io/docs/overview/what-is-zenoh/) as its ROS 2 middleware. Zenoh is the messaging layer that lets compatible ROS 2 nodes discover one another and exchange data between ROS 2 containers on the Duckiedrone. [`rmw_zenoh_cpp`](https://github.com/ros2/rmw_zenoh) is the ROS 2 adapter that lets ROS 2 use Zenoh. The RMW implementation variable, `RMW_IMPLEMENTATION`, selects that adapter. A `zenoh-router` container connects the ROS 2 containers, and standard ROS 2 services on a Duckiedrone use `ROS_DOMAIN_ID=42`.

### Try it

In the supplied ROS 2 exercise terminal, source ROS 2 and inspect your shell's domain and middleware settings and the nodes it can currently see. These read-only commands describe the local exercise environment, not a Duckiedrone.

```bash
source /opt/ros/${ROS2_DISTRO}/setup.bash
printf 'ROS_DOMAIN_ID=%s\n' "${ROS_DOMAIN_ID:-unset}"
printf 'RMW_IMPLEMENTATION=%s\n' "${RMW_IMPLEMENTATION:-unset}"
ros2 node list
```

An `unset` value means this shell has not explicitly selected that setting. The node list may be empty before starting a local node. Can you infer the Duckiedrone's domain or available nodes from this output?

<details>
<summary>Interpret the result</summary>
<p>No. The values and node list describe only this exercise shell and its visible graph; ROS 2 can use defaults when the variables are unset. Device settings must be checked in an authorized device shell, without reconfiguring the device to make its graph visible.</p>
</details>

A shell that inspects a ROS 2 graph running on a Duckiedrone must be able to reach the Duckiedrone's network and use a compatible middleware implementation and domain ID. A separate local experiment can use another domain when sharing a network with other ROS 2 users. To inspect a running Duckiedrone, retain its configured domain or use a ROS 2 shell on the Duckiedrone. Owner-authorized inspection is read-only: do not change its domain or middleware settings. [Notebook 8](./8-inspect-a-duckiedrone-ros-2-graph.ipynb) provides that device-specific workflow.

## Absolute, relative, and private names

The form of a name affects where it appears in the graph, as [Table 1](#table-1) shows.

<table id="table-1" class="lx-table">
  <caption>Table 1: ROS 2 name forms and their resolution.</caption>
  <thead>
    <tr>
      <th>Form</th>
      <th>Example</th>
      <th>Meaning</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>Absolute</td>
      <td><code>/DUCKIEDRONE_NAME/camera_node/camera_info</code></td>
      <td>The complete graph name, starting at the root <code>/</code></td>
    </tr>
    <tr>
      <td>Relative</td>
      <td><code>camera_info</code></td>
      <td>ROS 2 adds the node's namespace. For a node named <code>/DUCKIEDRONE_NAME/camera_node</code>, where <code>DUCKIEDRONE_NAME</code> is the name of your Duckiedrone, this becomes <code>/DUCKIEDRONE_NAME/camera_info</code></td>
    </tr>
    <tr>
      <td>Private</td>
      <td><code>~/state</code></td>
      <td>ROS 2 adds the node's namespace and name. For that same node, this becomes <code>/DUCKIEDRONE_NAME/camera_node/state</code></td>
    </tr>
  </tbody>
</table>

Private names are useful for a node's implementation details because they avoid collisions with another instance of the same node. Camera and bottom ToF bridge topics on the deployed ROS 2 graph commonly look like `/DUCKIEDRONE_NAME/camera_node/image/compressed` and `/DUCKIEDRONE_NAME/bottom_tof_driver_node/range`, while MAVROS, the bridge that exposes flight-controller data to ROS 2, has root-level topics such as `/mavros/imu/data`. Inspect the live graph to resolve topic names.

## If you have seen ROS 1

The basic ideas remain familiar, but this LX uses ROS 2 throughout. [Table 2](#table-2) is a translation reference, not a set of commands to mix into the LX.

<table id="table-2" class="lx-table">
  <caption>Table 2: ROS 1 concepts and their ROS 2 counterparts in this LX.</caption>
  <thead>
    <tr>
      <th>ROS 1</th>
      <th>ROS 2 in this LX</th>
      <th>Practical change</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><code>roscore</code> and a ROS master</td>
      <td>Middleware discovery, using Zenoh on Duckiedrones, and <code>ROS_DOMAIN_ID</code></td>
      <td>Do not start <code>roscore</code>; nodes discover compatible participants themselves</td>
    </tr>
    <tr>
      <td><code>catkin_make</code> and <code>devel/setup.bash</code></td>
      <td><code>colcon build</code> and <code>install/setup.bash</code></td>
      <td>Build the workspace, then source its installed overlay</td>
    </tr>
    <tr>
      <td><code>rospy</code> and <code>rospy.Rate</code></td>
      <td><code>rclpy</code>, nodes, timers, and executors</td>
      <td>Timers and subscriptions invoke callbacks while <code>rclpy.spin()</code> runs</td>
    </tr>
    <tr>
      <td><code>rosrun</code> and <code>rostopic</code></td>
      <td><code>ros2 run</code> and <code>ros2 topic</code></td>
      <td>The graph-inspection goal is the same, but the commands differ</td>
    </tr>
    <tr>
      <td><code>anonymous=True</code></td>
      <td>Explicit names and command-line remapping</td>
      <td>Choose a unique name or remap one when starting another node</td>
    </tr>
  </tbody>
</table>

For this LX, use the ROS 2 commands and APIs shown in the later notebooks.

## Further reading

The ROS 2 Jazzy documentation on [names](https://docs.ros.org/en/jazzy/Concepts/Basic/About-Names.html), [`ROS_DOMAIN_ID`](https://docs.ros.org/en/jazzy/Concepts/Basic/About-Domain-ID.html), and [Quality of Service settings](https://docs.ros.org/en/jazzy/Concepts/Basic/About-Quality-of-Service-Settings.html) expands on these graph rules.

## Checkpoint

Run the self-check in the next cell. Write or select a response before revealing the answer.


In [ ]:
import sys
from pathlib import Path

working_directory = Path.cwd()
parent_directory = working_directory.parent
if (parent_directory / "packages").is_dir():
    parent_directory_path = str(parent_directory)
    sys.path.insert(0, parent_directory_path)

from packages.checkpoint_self_check import display_checkpoint_self_checks

display_checkpoint_self_checks()
